In [ ]:
import geopandas as gpd
import numpy as np
import xarray as xr
import pandas as pd
from shapely.geometry import MultiPolygon, Polygon, Point
from matplotlib.path import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as colors
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import seaborn as sns
import datetime
import cmocean
import scipy
import calendar

In [ ]:
# Load and read shapefile

epu_shp = r'\Users\ratna\OneDrive\Documents\NPP\EPUs\EPUs.shp'
gdf = gpd.read_file(epu_shp)

# print(gdf.head())

In [ ]:
# Re-project the shapefile
gdf = gdf.to_crs(epsg=4326)

In [ ]:
# Get unique values from EPU and geometry values

name = gdf['Name'].unique()
geometry = gdf['geometry'].unique()

In [ ]:
# Load datasets

path = r'\Users\ratna\OneDrive\Documents\NPP\Temporary output\npp_chl1.nc'
npp_chl = xr.open_dataset(path)

In [ ]:
# Compute monthly climatology for NPP VGPM
npp_vgpm = npp_chl['VGPM']
vgpm = npp_vgpm.groupby('time.month').mean(skipna = True, dim=['time'])/1000  # unit in grC/m2/day

In [ ]:
# Compute monthly climatology for NPP VGPM
npp_evgpm = npp_chl['EVGPM']
evgpm = npp_evgpm.groupby('time.month').mean(skipna = True, dim=['time'])/1000  # unit in grC/m2/day

#evgpm = npp_evgpm.groupby('time.month').mean(skipna = True, dim=['time'])/1000  # unit in grC/m2/day >replace by std

In [ ]:
# Initialize coords for NPP chl
lat = npp_chl['latitude'].values
lon = npp_chl['longitude'].values

In [ ]:
# Create dictionary of name and geometry for all EPU

epus = dict(zip(name, geometry))
epus

In [ ]:
# Create function to handle both Polygon or Multipolygons --> extract the exterior coordinates from a given geometry object

def get_polygon_coords(geometry):
    # If the geometry is polygon
    if isinstance(geometry, Polygon):
        return [(coord[0], coord[1]) for coord in geometry.exterior.coords]
    
    # If the geometry is multipolygon
    elif isinstance(geometry, MultiPolygon):
        coords = []
        for poly in geometry.geoms:  # Use .geoms to access each polygon
            coords.extend([(coord[0], coord[1]) for coord in poly.exterior.coords])
        return coords

In [ ]:
# Apply get_polygon_coords function to each EPU (to create polygons for each EPU)

Labrador_Shelf_coords = get_polygon_coords(epus['Labrador Shelf'])
Labrador_Shelf = Polygon(Labrador_Shelf_coords)

NE_Newfoundland_Shelf_coords = get_polygon_coords(epus['Northeast Newfoundland Shelf'])
NE_Newfoundland_Shelf = Polygon(NE_Newfoundland_Shelf_coords)

Grand_Banks_coords = get_polygon_coords(epus['Grand Banks'])
Grand_Banks = Polygon(Grand_Banks_coords)

Scotian_Shelf_coords = get_polygon_coords(epus['Scotian Shelf'])
Scotian_Shelf = Polygon(Scotian_Shelf_coords)

Gulf_of_Maine_coords = get_polygon_coords(epus['Gulf of Maine'])
Gulf_of_Maine = Polygon(Gulf_of_Maine_coords)

Georges_Bank_coords = get_polygon_coords(epus["George's Bank"])
Georges_Bank = Polygon(Georges_Bank_coords)

Mid_Atlantic_Bight_coords = get_polygon_coords(epus['Mid-Atlantic Bight'])
Mid_Atlantic_Bight = Polygon(Mid_Atlantic_Bight_coords)

Southern_Newfoundland_coords = get_polygon_coords(epus['Southern Newfoundland'])
Southern_Newfoundland = Polygon(Southern_Newfoundland_coords)

Flemish_Cap_coords = get_polygon_coords(epus['Flemish Cap'])
Flemish_Cap = Polygon(Flemish_Cap_coords)

In [ ]:
# Create mask arrays for each EPU to track grid points inside of the polygon --> need latitude and longitude from dataset

# NPP from chl
Labrador_Shelf_mask = np.zeros((len(lat), len(lon)))*np.nan
NE_Newfoundland_Shelf_mask = np.zeros((len(lat), len(lon)))*np.nan
Grand_Banks_mask = np.zeros((len(lat), len(lon)))*np.nan
Scotian_Shelf_mask = np.zeros((len(lat), len(lon)))*np.nan
Gulf_of_Maine_mask = np.zeros((len(lat), len(lon)))*np.nan
Georges_Bank_mask = np.zeros((len(lat), len(lon)))*np.nan
Mid_Atlantic_Bight_mask = np.zeros((len(lat), len(lon)))*np.nan
Southern_Newfoundland_mask = np.zeros((len(lat), len(lon)))*np.nan
Flemish_Cap_mask = np.zeros((len(lat), len(lon)))*np.nan

In [ ]:
# Counter

Labrador_Shelf_counter = 0
NE_Newfoundland_Shelf_counter = 0
Grand_Banks_counter = 0
Scotian_Shelf_counter = 0
Gulf_of_Maine_counter = 0
Georges_Bank_counter = 0
Mid_Atlantic_Bight_counter = 0
Southern_Newfoundland_counter = 0
Flemish_Cap_counter = 0

In [ ]:
# Loop through all grid points and mask those inside the polygons

# select data in polygon (for npp_chl's coords)
for i, xx in enumerate(lon):
    for j, yy in enumerate(lat):
        point = Point(lon[i], lat[j])
        if Labrador_Shelf.contains(point):
            Labrador_Shelf_counter = Labrador_Shelf_counter + 1
            Labrador_Shelf_mask[j,i] = 0
        elif NE_Newfoundland_Shelf.contains(point):
            NE_Newfoundland_Shelf_counter = NE_Newfoundland_Shelf_counter + 1
            NE_Newfoundland_Shelf_mask[j,i] = 0
        elif Grand_Banks.contains(point):
            Grand_Banks_counter = Grand_Banks_counter + 1
            Grand_Banks_mask[j,i] = 0
        elif Scotian_Shelf.contains(point):
            Scotian_Shelf_counter = Scotian_Shelf_counter + 1
            Scotian_Shelf_mask[j,i] = 0
        elif Gulf_of_Maine.contains(point):
            Gulf_of_Maine_counter = Gulf_of_Maine_counter + 1
            Gulf_of_Maine_mask[j,i] = 0
        elif Georges_Bank.contains(point):
            Georges_Bank_counter = Georges_Bank_counter + 1
            Georges_Bank_mask[j,i] = 0
        elif Mid_Atlantic_Bight.contains(point):
            Mid_Atlantic_Bight_counter = Mid_Atlantic_Bight_counter + 1
            Mid_Atlantic_Bight_mask[j,i] = 0
        elif Southern_Newfoundland.contains(point):
            Southern_Newfoundland_counter = Southern_Newfoundland_counter + 1
            Southern_Newfoundland_mask[j,i] = 0
        elif Flemish_Cap.contains(point):
            Flemish_Cap_counter = Flemish_Cap_counter + 1
            Flemish_Cap_mask[j,i] = 0        

In [ ]:
y1_Labrador_Shelf, x1_Labrador_Shelf = np.where(Labrador_Shelf_mask==0)
y1_NE_Newfoundland_Shelf, x1_NE_Newfoundland_Shelf = np.where(NE_Newfoundland_Shelf_mask==0)
y1_Grand_Banks, x1_Grand_Banks = np.where(Grand_Banks_mask==0)
y1_Scotian_Shelf, x1_Scotian_Shelf = np.where(Scotian_Shelf_mask==0)
y1_Gulf_of_Maine, x1_Gulf_of_Maine = np.where(Gulf_of_Maine_mask==0)
y1_Georges_Bank, x1_Georges_Bank = np.where(Georges_Bank_mask==0)
y1_Mid_Atlantic_Bight, x1_Mid_Atlantic_Bight = np.where(Mid_Atlantic_Bight_mask==0)
y1_Southern_Newfoundland, x1_Southern_Newfoundland = np.where(Southern_Newfoundland_mask==0)
y1_Flemish_Cap, x1_Flemish_Cap = np.where(Flemish_Cap_mask==0)

In [ ]:
vgpm_Labrador_Shelf = vgpm.isel(longitude=x1_Labrador_Shelf, latitude=y1_Labrador_Shelf)
print(vgpm_Labrador_Shelf.shape)
lon_Labrador_Shelf = vgpm_Labrador_Shelf.longitude.values
lat_Labrador_Shelf = vgpm_Labrador_Shelf.latitude.values
df_Labrador_Shelf = vgpm_Labrador_Shelf.mean(('latitude', 'longitude')).to_pandas()
del vgpm_Labrador_Shelf

vgpm_NE_Newfoundland_Shelf = vgpm.isel(longitude=x1_NE_Newfoundland_Shelf, latitude=y1_NE_Newfoundland_Shelf)
print(vgpm_NE_Newfoundland_Shelf.shape)
lon_NE_Newfoundland_Shelf = vgpm_NE_Newfoundland_Shelf.longitude.values
lat_NE_Newfoundland_Shelf = vgpm_NE_Newfoundland_Shelf.latitude.values
df_NE_Newfoundland_Shelf = vgpm_NE_Newfoundland_Shelf.mean(('latitude', 'longitude')).to_pandas()
del vgpm_NE_Newfoundland_Shelf

vgpm_Grand_Banks = vgpm.isel(longitude=x1_Grand_Banks, latitude=y1_Grand_Banks)
print(vgpm_Grand_Banks.shape)
lon_Grand_Banks = vgpm_Grand_Banks.longitude.values
lat_Grand_Banks = vgpm_Grand_Banks.latitude.values
df_Grand_Banks = vgpm_Grand_Banks.mean(('latitude', 'longitude')).to_pandas()
del vgpm_Grand_Banks

vgpm_Scotian_Shelf = vgpm.isel(longitude=x1_Scotian_Shelf, latitude=y1_Scotian_Shelf)
print(vgpm_Scotian_Shelf.shape)
lon_Scotian_Shelf = vgpm_Scotian_Shelf.longitude.values
lat_Scotian_Shelf = vgpm_Scotian_Shelf.latitude.values
df_Scotian_Shelf = vgpm_Scotian_Shelf.mean(('latitude', 'longitude')).to_pandas()
del vgpm_Scotian_Shelf

vgpm_Gulf_of_Maine = vgpm.isel(longitude=x1_Gulf_of_Maine, latitude=y1_Gulf_of_Maine)
print(vgpm_Gulf_of_Maine.shape)
lon_Gulf_of_Maine = vgpm_Gulf_of_Maine.longitude.values
lat_Gulf_of_Maine = vgpm_Gulf_of_Maine.latitude.values
df_Gulf_of_Maine = vgpm_Gulf_of_Maine.mean(('latitude', 'longitude')).to_pandas()
del vgpm_Gulf_of_Maine

vgpm_Georges_Bank = vgpm.isel(longitude=x1_Georges_Bank, latitude=y1_Georges_Bank)
print(vgpm_Georges_Bank.shape)
lon_Georges_Bank = vgpm_Georges_Bank.longitude.values
lat_Georges_Bank = vgpm_Georges_Bank.latitude.values
df_Georges_Bank = vgpm_Georges_Bank.mean(('latitude', 'longitude')).to_pandas()
del vgpm_Georges_Bank

vgpm_Mid_Atlantic_Bight = vgpm.isel(longitude=x1_Mid_Atlantic_Bight, latitude=y1_Mid_Atlantic_Bight)
print(vgpm_Mid_Atlantic_Bight.shape)
lon_Mid_Atlantic_Bight = vgpm_Mid_Atlantic_Bight.longitude.values
lat_Mid_Atlantic_Bight = vgpm_Mid_Atlantic_Bight.latitude.values
df_Mid_Atlantic_Bight = vgpm_Mid_Atlantic_Bight.mean(('latitude', 'longitude')).to_pandas()
del vgpm_Mid_Atlantic_Bight

vgpm_Southern_Newfoundland = vgpm.isel(longitude=x1_Southern_Newfoundland, latitude=y1_Southern_Newfoundland)
print(vgpm_Southern_Newfoundland.shape)
lon_Southern_Newfoundland = vgpm_Southern_Newfoundland.longitude.values
lat_Southern_Newfoundland = vgpm_Southern_Newfoundland.latitude.values
df_Southern_Newfoundland = vgpm_Southern_Newfoundland.mean(('latitude', 'longitude')).to_pandas()
del vgpm_Southern_Newfoundland

vgpm_Flemish_Cap = vgpm.isel(longitude=x1_Flemish_Cap, latitude=y1_Flemish_Cap)
print(vgpm_Flemish_Cap.shape)
lon_Flemish_Cap = vgpm_Flemish_Cap.longitude.values
lat_Flemish_Cap = vgpm_Flemish_Cap.latitude.values
df_Flemish_Cap = vgpm_Flemish_Cap.mean(('latitude', 'longitude')).to_pandas()
del vgpm_Flemish_Cap

In [ ]:
## ---- Concat and Save data ---- ##
# Concat
monthly_vgpm = pd.concat([df_Labrador_Shelf, df_NE_Newfoundland_Shelf, df_Grand_Banks, df_Scotian_Shelf, df_Gulf_of_Maine, df_Georges_Bank, df_Mid_Atlantic_Bight, df_Southern_Newfoundland, df_Flemish_Cap], keys=['Labrador_Shelf', 'NE_Newfoundland_Shelf', 'Grand_Banks', 'Scotian_Shelf', 'Gulf_of_Maine', 'Georges_Bank', 'Mid_Atlantic_Bight', 'Southern_Newfoundland', 'Flemish_Cap'], axis=1)
monthly_vgpm.index = [calendar.month_abbr[m] for m in monthly_vgpm.index]

month_order = list(calendar.month_abbr)[1:]  # ['Jan', ..., 'Dec']
monthly_vgpm.index = pd.CategoricalIndex(monthly_vgpm.index, categories=month_order, ordered=True)
monthly_vgpm = monthly_vgpm.sort_index()

#annual_hind.to_csv('annual_hind.csv', index=True)
print(monthly_vgpm)

In [ ]:
monthly_vgpm['Product'] = "VGPM"

In [ ]:
monthly_vgpm.reset_index()

In [ ]:
evgpm_Labrador_Shelf = evgpm.isel(longitude=x1_Labrador_Shelf, latitude=y1_Labrador_Shelf)
print(evgpm_Labrador_Shelf.shape)
lon_Labrador_Shelf = evgpm_Labrador_Shelf.longitude.values
lat_Labrador_Shelf = evgpm_Labrador_Shelf.latitude.values
df_Labrador_Shelf1 = evgpm_Labrador_Shelf.mean(('latitude', 'longitude')).to_pandas()
del evgpm_Labrador_Shelf

evgpm_NE_Newfoundland_Shelf = evgpm.isel(longitude=x1_NE_Newfoundland_Shelf, latitude=y1_NE_Newfoundland_Shelf)
print(evgpm_NE_Newfoundland_Shelf.shape)
lon_NE_Newfoundland_Shelf = evgpm_NE_Newfoundland_Shelf.longitude.values
lat_NE_Newfoundland_Shelf = evgpm_NE_Newfoundland_Shelf.latitude.values
df_NE_Newfoundland_Shelf1 = evgpm_NE_Newfoundland_Shelf.mean(('latitude', 'longitude')).to_pandas()
del evgpm_NE_Newfoundland_Shelf

evgpm_Grand_Banks = evgpm.isel(longitude=x1_Grand_Banks, latitude=y1_Grand_Banks)
print(evgpm_Grand_Banks.shape)
lon_Grand_Banks = evgpm_Grand_Banks.longitude.values
lat_Grand_Banks = evgpm_Grand_Banks.latitude.values
df_Grand_Banks1 = evgpm_Grand_Banks.mean(('latitude', 'longitude')).to_pandas()
del evgpm_Grand_Banks

evgpm_Scotian_Shelf = evgpm.isel(longitude=x1_Scotian_Shelf, latitude=y1_Scotian_Shelf)
print(evgpm_Scotian_Shelf.shape)
lon_Scotian_Shelf = evgpm_Scotian_Shelf.longitude.values
lat_Scotian_Shelf = evgpm_Scotian_Shelf.latitude.values
df_Scotian_Shelf1 = evgpm_Scotian_Shelf.mean(('latitude', 'longitude')).to_pandas()
del evgpm_Scotian_Shelf

evgpm_Gulf_of_Maine = evgpm.isel(longitude=x1_Gulf_of_Maine, latitude=y1_Gulf_of_Maine)
print(evgpm_Gulf_of_Maine.shape)
lon_Gulf_of_Maine = evgpm_Gulf_of_Maine.longitude.values
lat_Gulf_of_Maine = evgpm_Gulf_of_Maine.latitude.values
df_Gulf_of_Maine1 = evgpm_Gulf_of_Maine.mean(('latitude', 'longitude')).to_pandas()
del evgpm_Gulf_of_Maine

evgpm_Georges_Bank = evgpm.isel(longitude=x1_Georges_Bank, latitude=y1_Georges_Bank)
print(evgpm_Georges_Bank.shape)
lon_Georges_Bank = evgpm_Georges_Bank.longitude.values
lat_Georges_Bank = evgpm_Georges_Bank.latitude.values
df_Georges_Bank1 = evgpm_Georges_Bank.mean(('latitude', 'longitude')).to_pandas()
del evgpm_Georges_Bank

evgpm_Mid_Atlantic_Bight = evgpm.isel(longitude=x1_Mid_Atlantic_Bight, latitude=y1_Mid_Atlantic_Bight)
print(evgpm_Mid_Atlantic_Bight.shape)
lon_Mid_Atlantic_Bight = evgpm_Mid_Atlantic_Bight.longitude.values
lat_Mid_Atlantic_Bight = evgpm_Mid_Atlantic_Bight.latitude.values
df_Mid_Atlantic_Bight1 = evgpm_Mid_Atlantic_Bight.mean(('latitude', 'longitude')).to_pandas()
del evgpm_Mid_Atlantic_Bight

evgpm_Southern_Newfoundland = evgpm.isel(longitude=x1_Southern_Newfoundland, latitude=y1_Southern_Newfoundland)
print(evgpm_Southern_Newfoundland.shape)
lon_Southern_Newfoundland = evgpm_Southern_Newfoundland.longitude.values
lat_Southern_Newfoundland = evgpm_Southern_Newfoundland.latitude.values
df_Southern_Newfoundland1 = evgpm_Southern_Newfoundland.mean(('latitude', 'longitude')).to_pandas()
del evgpm_Southern_Newfoundland

evgpm_Flemish_Cap = evgpm.isel(longitude=x1_Flemish_Cap, latitude=y1_Flemish_Cap)
print(evgpm_Flemish_Cap.shape)
lon_Flemish_Cap = evgpm_Flemish_Cap.longitude.values
lat_Flemish_Cap = evgpm_Flemish_Cap.latitude.values
df_Flemish_Cap1 = evgpm_Flemish_Cap.mean(('latitude', 'longitude')).to_pandas()
del evgpm_Flemish_Cap

In [ ]:
## ---- Concat and Save data ---- ##
# Concat
monthly_evgpm = pd.concat([df_Labrador_Shelf1, df_NE_Newfoundland_Shelf1, df_Grand_Banks1, df_Scotian_Shelf1, df_Gulf_of_Maine1, df_Georges_Bank1, df_Mid_Atlantic_Bight1, df_Southern_Newfoundland1, df_Flemish_Cap1], keys=['Labrador_Shelf', 'NE_Newfoundland_Shelf', 'Grand_Banks', 'Scotian_Shelf', 'Gulf_of_Maine', 'Georges_Bank', 'Mid_Atlantic_Bight', 'Southern_Newfoundland', 'Flemish_Cap'], axis=1)
monthly_evgpm.index = [calendar.month_abbr[m] for m in monthly_evgpm.index]

month_order = list(calendar.month_abbr)[1:]  # ['Jan', ..., 'Dec']
monthly_evgpm.index = pd.CategoricalIndex(monthly_evgpm.index, categories=month_order, ordered=True)
monthly_evgpm = monthly_evgpm.sort_index()

#annual_hind.to_csv('annual_hind.csv', index=True)
print(monthly_evgpm)

In [ ]:
monthly_evgpm['Product'] = "EVGPM"

In [ ]:
monthly_evgpm.reset_index()

In [ ]:
path1 = r'\Users\ratna\OneDrive\Documents\NPP\Temporary output\npp_mod1.nc'
npp_mod = xr.open_dataset(path1)

In [ ]:
# Initialize coords for NPP mod
lats = npp_mod['latitude'].values
lons = npp_mod['longitude'].values

In [ ]:
# Compute annual net for NPP VGPM
npp_fore = npp_mod['FORE']
fore = npp_fore.groupby('time.month').mean(skipna = True, dim=['time'])/1000  # unit in grC/m2/day

In [ ]:
# Compute annual net for NPP VGPM
npp_hind = npp_mod['HIND']
hind = npp_hind.groupby('time.month').mean(skipna = True, dim=['time'])/1000  # unit in grC/m2/day

In [ ]:
# NPP from model

Labrador_Shelf_masks = np.zeros((len(lats), len(lons)))*np.nan
NE_Newfoundland_Shelf_masks = np.zeros((len(lats), len(lons)))*np.nan
Grand_Banks_masks = np.zeros((len(lats), len(lons)))*np.nan
Scotian_Shelf_masks = np.zeros((len(lats), len(lons)))*np.nan
Gulf_of_Maine_masks = np.zeros((len(lats), len(lons)))*np.nan
Georges_Bank_masks = np.zeros((len(lats), len(lons)))*np.nan
Mid_Atlantic_Bight_masks = np.zeros((len(lats), len(lons)))*np.nan
Southern_Newfoundland_masks = np.zeros((len(lats), len(lons)))*np.nan
Flemish_Cap_masks = np.zeros((len(lats), len(lons)))*np.nan

In [ ]:
# Loop through all grid points and mask those inside the polygons

# select data in polygon (for npp_mod's coords)
for i, xx in enumerate(lons):
    for j, yy in enumerate(lats):
        point = Point(lons[i], lats[j])
        if Labrador_Shelf.contains(point):
            Labrador_Shelf_counters = Labrador_Shelf_counter + 1
            Labrador_Shelf_masks[j,i] = 0
        elif NE_Newfoundland_Shelf.contains(point):
            NE_Newfoundland_Shelf_counters = NE_Newfoundland_Shelf_counter + 1
            NE_Newfoundland_Shelf_masks[j,i] = 0
        elif Grand_Banks.contains(point):
            Grand_Banks_counters = Grand_Banks_counter + 1
            Grand_Banks_masks[j,i] = 0
        elif Scotian_Shelf.contains(point):
            Scotian_Shelf_counters = Scotian_Shelf_counter + 1
            Scotian_Shelf_masks[j,i] = 0
        elif Gulf_of_Maine.contains(point):
            Gulf_of_Maine_counters = Gulf_of_Maine_counter + 1
            Gulf_of_Maine_masks[j,i] = 0
        elif Georges_Bank.contains(point):
            Georges_Bank_counters = Georges_Bank_counter + 1
            Georges_Bank_masks[j,i] = 0
        elif Mid_Atlantic_Bight.contains(point):
            Mid_Atlantic_Bight_counters = Mid_Atlantic_Bight_counter + 1
            Mid_Atlantic_Bight_masks[j,i] = 0
        elif Southern_Newfoundland.contains(point):
            Southern_Newfoundland_counters = Southern_Newfoundland_counter + 1
            Southern_Newfoundland_masks[j,i] = 0
        elif Flemish_Cap.contains(point):
            Flemish_Cap_counters = Flemish_Cap_counter + 1
            Flemish_Cap_masks[j,i] = 0        

In [ ]:
y2_Labrador_Shelf, x2_Labrador_Shelf = np.where(Labrador_Shelf_masks==0)
y2_NE_Newfoundland_Shelf, x2_NE_Newfoundland_Shelf = np.where(NE_Newfoundland_Shelf_masks==0)
y2_Grand_Banks, x2_Grand_Banks = np.where(Grand_Banks_masks==0)
y2_Scotian_Shelf, x2_Scotian_Shelf = np.where(Scotian_Shelf_masks==0)
y2_Gulf_of_Maine, x2_Gulf_of_Maine = np.where(Gulf_of_Maine_masks==0)
y2_Georges_Bank, x2_Georges_Bank = np.where(Georges_Bank_masks==0)
y2_Mid_Atlantic_Bight, x2_Mid_Atlantic_Bight = np.where(Mid_Atlantic_Bight_masks==0)
y2_Southern_Newfoundland, x2_Southern_Newfoundland = np.where(Southern_Newfoundland_masks==0)
y2_Flemish_Cap, x2_Flemish_Cap = np.where(Flemish_Cap_masks==0)

In [ ]:
fore_Labrador_Shelf = fore.isel(longitude=x2_Labrador_Shelf, latitude=y2_Labrador_Shelf)
print(fore_Labrador_Shelf.shape)
lon_Labrador_Shelf = fore_Labrador_Shelf.longitude.values
lat_Labrador_Shelf = fore_Labrador_Shelf.latitude.values
df_Labrador_Shelf2 = fore_Labrador_Shelf.mean(('latitude', 'longitude')).to_pandas()
del fore_Labrador_Shelf

fore_NE_Newfoundland_Shelf = fore.isel(longitude=x2_NE_Newfoundland_Shelf, latitude=y2_NE_Newfoundland_Shelf)
print(fore_NE_Newfoundland_Shelf.shape)
lon_NE_Newfoundland_Shelf = fore_NE_Newfoundland_Shelf.longitude.values
lat_NE_Newfoundland_Shelf = fore_NE_Newfoundland_Shelf.latitude.values
df_NE_Newfoundland_Shelf2 = fore_NE_Newfoundland_Shelf.mean(('latitude', 'longitude')).to_pandas()
del fore_NE_Newfoundland_Shelf

fore_Grand_Banks = fore.isel(longitude=x2_Grand_Banks, latitude=y2_Grand_Banks)
print(fore_Grand_Banks.shape)
lon_Grand_Banks = fore_Grand_Banks.longitude.values
lat_Grand_Banks = fore_Grand_Banks.latitude.values
df_Grand_Banks2 = fore_Grand_Banks.mean(('latitude', 'longitude')).to_pandas()
del fore_Grand_Banks

fore_Scotian_Shelf = fore.isel(longitude=x2_Scotian_Shelf, latitude=y2_Scotian_Shelf)
print(fore_Scotian_Shelf.shape)
lon_Scotian_Shelf = fore_Scotian_Shelf.longitude.values
lat_Scotian_Shelf = fore_Scotian_Shelf.latitude.values
df_Scotian_Shelf2 = fore_Scotian_Shelf.mean(('latitude', 'longitude')).to_pandas()
del fore_Scotian_Shelf

fore_Gulf_of_Maine = fore.isel(longitude=x2_Gulf_of_Maine, latitude=y2_Gulf_of_Maine)
print(fore_Gulf_of_Maine.shape)
lon_Gulf_of_Maine = fore_Gulf_of_Maine.longitude.values
lat_Gulf_of_Maine = fore_Gulf_of_Maine.latitude.values
df_Gulf_of_Maine2 = fore_Gulf_of_Maine.mean(('latitude', 'longitude')).to_pandas()
del fore_Gulf_of_Maine

fore_Georges_Bank = fore.isel(longitude=x2_Georges_Bank, latitude=y2_Georges_Bank)
print(fore_Georges_Bank.shape)
lon_Georges_Bank = fore_Georges_Bank.longitude.values
lat_Georges_Bank = fore_Georges_Bank.latitude.values
df_Georges_Bank2 = fore_Georges_Bank.mean(('latitude', 'longitude')).to_pandas()
del fore_Georges_Bank

fore_Mid_Atlantic_Bight = fore.isel(longitude=x2_Mid_Atlantic_Bight, latitude=y2_Mid_Atlantic_Bight)
print(fore_Mid_Atlantic_Bight.shape)
lon_Mid_Atlantic_Bight = fore_Mid_Atlantic_Bight.longitude.values
lat_Mid_Atlantic_Bight = fore_Mid_Atlantic_Bight.latitude.values
df_Mid_Atlantic_Bight2 = fore_Mid_Atlantic_Bight.mean(('latitude', 'longitude')).to_pandas()
del fore_Mid_Atlantic_Bight

fore_Southern_Newfoundland = fore.isel(longitude=x2_Southern_Newfoundland, latitude=y2_Southern_Newfoundland)
print(fore_Southern_Newfoundland.shape)
lon_Southern_Newfoundland = fore_Southern_Newfoundland.longitude.values
lat_Southern_Newfoundland = fore_Southern_Newfoundland.latitude.values
df_Southern_Newfoundland2 = fore_Southern_Newfoundland.mean(('latitude', 'longitude')).to_pandas()
del fore_Southern_Newfoundland

fore_Flemish_Cap = fore.isel(longitude=x2_Flemish_Cap, latitude=y2_Flemish_Cap)
print(fore_Flemish_Cap.shape)
lon_Flemish_Cap = fore_Flemish_Cap.longitude.values
lat_Flemish_Cap = fore_Flemish_Cap.latitude.values
df_Flemish_Cap2 = fore_Flemish_Cap.mean(('latitude', 'longitude')).to_pandas()
del fore_Flemish_Cap

In [ ]:
## ---- Concat and Save data ---- ##
# Concat
monthly_fore = pd.concat([df_Labrador_Shelf2, df_NE_Newfoundland_Shelf2, df_Grand_Banks2, df_Scotian_Shelf2, df_Gulf_of_Maine2, df_Georges_Bank2, df_Mid_Atlantic_Bight2, df_Southern_Newfoundland2, df_Flemish_Cap2], keys=['Labrador_Shelf', 'NE_Newfoundland_Shelf', 'Grand_Banks', 'Scotian_Shelf', 'Gulf_of_Maine', 'Georges_Bank', 'Mid_Atlantic_Bight', 'Southern_Newfoundland', 'Flemish_Cap'], axis=1)
monthly_fore.index = [calendar.month_abbr[m] for m in monthly_fore.index]

month_order = list(calendar.month_abbr)[1:]  # ['Jan', ..., 'Dec']
monthly_fore.index = pd.CategoricalIndex(monthly_fore.index, categories=month_order, ordered=True)
monthly_fore = monthly_fore.sort_index()

#annual_hind.to_csv('annual_hind.csv', index=True)
print(monthly_fore)

In [ ]:
monthly_fore['Product']="Forecast"

In [ ]:
monthly_fore.reset_index()

In [ ]:
hind_Labrador_Shelf = hind.isel(longitude=x2_Labrador_Shelf, latitude=y2_Labrador_Shelf)
print(hind_Labrador_Shelf.shape)
lon_Labrador_Shelf = hind_Labrador_Shelf.longitude.values
lat_Labrador_Shelf = hind_Labrador_Shelf.latitude.values
df_Labrador_Shelf3 = hind_Labrador_Shelf.mean(('latitude', 'longitude')).to_pandas()
del hind_Labrador_Shelf

hind_NE_Newfoundland_Shelf = hind.isel(longitude=x2_NE_Newfoundland_Shelf, latitude=y2_NE_Newfoundland_Shelf)
print(hind_NE_Newfoundland_Shelf.shape)
lon_NE_Newfoundland_Shelf = hind_NE_Newfoundland_Shelf.longitude.values
lat_NE_Newfoundland_Shelf = hind_NE_Newfoundland_Shelf.latitude.values
df_NE_Newfoundland_Shelf3 = hind_NE_Newfoundland_Shelf.mean(('latitude', 'longitude')).to_pandas()
del hind_NE_Newfoundland_Shelf

hind_Grand_Banks = hind.isel(longitude=x2_Grand_Banks, latitude=y2_Grand_Banks)
print(hind_Grand_Banks.shape)
lon_Grand_Banks = hind_Grand_Banks.longitude.values
lat_Grand_Banks = hind_Grand_Banks.latitude.values
df_Grand_Banks3 = hind_Grand_Banks.mean(('latitude', 'longitude')).to_pandas()
del hind_Grand_Banks

hind_Scotian_Shelf = hind.isel(longitude=x2_Scotian_Shelf, latitude=y2_Scotian_Shelf)
print(hind_Scotian_Shelf.shape)
lon_Scotian_Shelf = hind_Scotian_Shelf.longitude.values
lat_Scotian_Shelf = hind_Scotian_Shelf.latitude.values
df_Scotian_Shelf3 = hind_Scotian_Shelf.mean(('latitude', 'longitude')).to_pandas()
del hind_Scotian_Shelf

hind_Gulf_of_Maine = hind.isel(longitude=x2_Gulf_of_Maine, latitude=y2_Gulf_of_Maine)
print(hind_Gulf_of_Maine.shape)
lon_Gulf_of_Maine = hind_Gulf_of_Maine.longitude.values
lat_Gulf_of_Maine = hind_Gulf_of_Maine.latitude.values
df_Gulf_of_Maine3 = hind_Gulf_of_Maine.mean(('latitude', 'longitude')).to_pandas()
del hind_Gulf_of_Maine

hind_Georges_Bank = hind.isel(longitude=x2_Georges_Bank, latitude=y2_Georges_Bank)
print(hind_Georges_Bank.shape)
lon_Georges_Bank = hind_Georges_Bank.longitude.values
lat_Georges_Bank = hind_Georges_Bank.latitude.values
df_Georges_Bank3 = hind_Georges_Bank.mean(('latitude', 'longitude')).to_pandas()
del hind_Georges_Bank

hind_Mid_Atlantic_Bight = hind.isel(longitude=x2_Mid_Atlantic_Bight, latitude=y2_Mid_Atlantic_Bight)
print(hind_Mid_Atlantic_Bight.shape)
lon_Mid_Atlantic_Bight = hind_Mid_Atlantic_Bight.longitude.values
lat_Mid_Atlantic_Bight = hind_Mid_Atlantic_Bight.latitude.values
df_Mid_Atlantic_Bight3 = hind_Mid_Atlantic_Bight.mean(('latitude', 'longitude')).to_pandas()
del hind_Mid_Atlantic_Bight

hind_Southern_Newfoundland = hind.isel(longitude=x2_Southern_Newfoundland, latitude=y2_Southern_Newfoundland)
print(hind_Southern_Newfoundland.shape)
lon_Southern_Newfoundland = hind_Southern_Newfoundland.longitude.values
lat_Southern_Newfoundland = hind_Southern_Newfoundland.latitude.values
df_Southern_Newfoundland3 = hind_Southern_Newfoundland.mean(('latitude', 'longitude')).to_pandas()
del hind_Southern_Newfoundland

hind_Flemish_Cap = hind.isel(longitude=x2_Flemish_Cap, latitude=y2_Flemish_Cap)
print(hind_Flemish_Cap.shape)
lon_Flemish_Cap = hind_Flemish_Cap.longitude.values
lat_Flemish_Cap = hind_Flemish_Cap.latitude.values
df_Flemish_Cap3 = hind_Flemish_Cap.mean(('latitude', 'longitude')).to_pandas()
#del hind_Flemish_Cap

In [ ]:
## ---- Concat and Save data ---- ##
# Concat
monthly_hind = pd.concat([df_Labrador_Shelf3, df_NE_Newfoundland_Shelf3, df_Grand_Banks3, df_Scotian_Shelf3, df_Gulf_of_Maine3, df_Georges_Bank3, df_Mid_Atlantic_Bight3, df_Southern_Newfoundland3, df_Flemish_Cap3], keys=['Labrador_Shelf', 'NE_Newfoundland_Shelf', 'Grand_Banks', 'Scotian_Shelf', 'Gulf_of_Maine', 'Georges_Bank', 'Mid_Atlantic_Bight', 'Southern_Newfoundland', 'Flemish_Cap'], axis=1)
monthly_hind.index = [calendar.month_abbr[m] for m in monthly_hind.index]

month_order = list(calendar.month_abbr)[1:]  # ['Jan', ..., 'Dec']
monthly_hind.index = pd.CategoricalIndex(monthly_hind.index, categories=month_order, ordered=True)
monthly_hind = monthly_hind.sort_index()

#print(monthly_hind)
monthly_hind

In [ ]:
monthly_hind['Product']="Hindcast"

In [ ]:
monthly_hind.reset_index()

In [ ]:
month_order = ['Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May',
               'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov']

In [ ]:
# Define ncols and nrows

ncols = 3
nrows = 3

# Create figure and grid
fig, axes = plt.subplots(ncols = ncols, nrows=nrows, figsize=(37.5, 20))
plt.subplots_adjust(bottom=0.25, top=0.96, left=0.04, right=0.99, wspace=0.2, hspace=0.4)

# List the EPU name
epu_name = ['Labrador_Shelf', 'NE_Newfoundland_Shelf', 'Grand_Banks', 'Southern_Newfoundland', 'Flemish_Cap',
            'Scotian_Shelf', 'Gulf_of_Maine', 'Georges_Bank', 'Mid_Atlantic_Bight']

# Loop to draw graphs for each subplot
for i, place in enumerate(epu_name):
    ax = axes[i//3, i%3]  
    
    # Plot data for each EPU
    #ax.plot(monthly_vgpm.index, monthly_vgpm[place], label='VGPM', color = 'b')
    #ax.plot(monthly_evgpm.index, monthly_evgpm[place], label='Eppley VGPM', color = 'r', ls='--')
    #ax.plot(monthly_fore.index, monthly_fore[place], label='Forecast', color = 'g')
    #ax.plot(monthly_hind.index, monthly_hind[place], label='Hindcast', color = 'magenta')
    
    # Sort data per EPU based on customized months order
    vgpm = monthly_vgpm[place].reindex(month_order)
    evgpm = monthly_evgpm[place].reindex(month_order)
    fore = monthly_fore[place].reindex(month_order)
    hind = monthly_hind[place].reindex(month_order)

    # Plot
    ax.plot(month_order, vgpm, label='VGPM', color='b')
    ax.plot(month_order, evgpm, label='Eppley VGPM', color='r', ls='--')
    ax.plot(month_order, fore, label='Forecast', color='g')
    ax.plot(month_order, hind, label='Hindcast', color='magenta')
    
    # Add labels for each graph
    ax.set_title(place, fontsize=20)
    ax.set_xlabel('Month', fontsize=17, labelpad=12)
    ax.set_ylabel('NPP', fontsize=17, labelpad=12)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.grid(True)

fig.suptitle(r'Monthly Climatology of NPP ($\mathbf{gC}$ $\mathbf{m^{-2}}$ $\mathbf{day^{-1}}$)', fontsize=24, fontweight='bold', y=1.0)
plt.tight_layout()
plt.legend(loc='center', ncols=4, bbox_to_anchor=(-0.57, -0.20), fontsize=20)
#plt.savefig('Monthly climatology_300.png', format = 'png', dpi = 300, bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Pilih hanya 3 lokasi
epu_name = ['Labrador_Shelf', 'Flemish_Cap', 'Gulf_of_Maine']

# Layout subplot: 1 baris × 3 kolom
ncols = 1
nrows = 3

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(9, 18)) #30, 8
axes = axes.flatten()

for i, place in enumerate(epu_name):
    ax = axes[i]
    
    # Urutkan data sesuai month_order
    vgpm  = monthly_vgpm[place].reindex(month_order)
    evgpm = monthly_evgpm[place].reindex(month_order)
    fore  = monthly_fore[place].reindex(month_order)
    hind  = monthly_hind[place].reindex(month_order)

    # Plot tanpa legend
    ax.plot(month_order, vgpm,  color='b')
    ax.plot(month_order, evgpm, color='r', ls='--')
    ax.plot(month_order, fore,  color='g')
    ax.plot(month_order, hind, color='magenta')

    ax.set_title(place.replace('_', ' '), fontsize=20)
    ax.set_xlabel('Month', fontsize=16)
    ax.set_ylabel('NPP', fontsize=16)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.grid(True)

# Judul utama
#fig.suptitle(r'Monthly Climatology of NPP ($\mathbf{gC}$ $\mathbf{m^{-2}}$ $\mathbf{day^{-1}}$)', fontsize=24, fontweight='bold', y=1.02)

# Atur layout agar rapi
plt.tight_layout(rect=[0, 0.04, 1, 0.95])
#plt.savefig('Monthly300_notitle.png', format = 'png', dpi = 300, bbox_inches='tight')
plt.show()

In [ ]:
all_monthly = pd.concat([monthly_vgpm, monthly_evgpm, monthly_fore, monthly_hind]).reset_index()

all_monthly

In [ ]:
data_epu = r'C:\Users\ratna\OneDrive\Documents\NPP\Temporary output\EPU_area.xlsx'
epu_area = pd.read_excel(data_epu)

epu_area

In [ ]:
area_dict = dict(zip(epu_area['Unnamed: 0'], epu_area['Area (km\u00b2)']))

In [ ]:
# Copy df biar gak ubah asli
monthly_weighted = all_monthly.copy()

# Loop tiap kolom lokasi, dan kalikan dengan area
for col in monthly_weighted.columns:
    if col not in ['index', 'Product']:
        monthly_weighted[col] = (monthly_weighted[col] * area_dict.get(col, 1)) / np.sum(list(area_dict.values())) # 1 as fallback

In [ ]:
monthly_weighted.round(2)

In [ ]:
monthly_melt = monthly_weighted.melt(id_vars=['index', 'Product'], var_name='Location', value_name='Mean')

In [ ]:
monthly_melt

In [ ]:
monthly_clim = monthly_melt.pivot_table(index=['index', 'Product'], columns='Location', values='Mean').round(2)

In [ ]:
monthly_clim.rename_axis(index={'index': 'Month'}, inplace=True)

In [ ]:
monthly_clim

In [ ]:
monthly_clim['mean'] = monthly_clim.mean(axis=1).round(2)
monthly_clim['\u00B1 std'] = monthly_clim.std(axis=1).round(2)

In [ ]:
monthly_clim

In [ ]:
mean = monthly_clim.mean(axis=1).unstack(level=0)
std = monthly_clim.std(axis=1).unstack(level=0)

In [ ]:
mean['Stat']='Mean'
std['Stat']='\u00B1 Std'

In [ ]:
product_sum = pd.concat([mean, std]).sort_index()

In [ ]:
product_sum.round(2)

In [ ]:
product_sum.set_index('Stat', append=True).round(2).to_excel(r'C:\Users\ratna\OneDrive\Documents\NPP\Temporary output\climatology1.xlsx')